# Orthogonal polynomials, then ridge

On $x = -2, -1, 0, 1, 2$ the column $x$ is already orthogonal to the intercept. The column $x^2$ is not: its mean is $2$. Subtracting that mean produces a curvature column orthogonal to both lower columns, and each coefficient becomes a single division. Ridge shrinkage of those orthogonal coefficients is then one fraction per column.


## The centered square

The values of $x^2$ are $4, 1, 0, 1, 4$, with mean $10/5 = 2$. Define

$$
p_2(x) = x^2 - 2,
$$

so the column is $2, -1, -2, -1, 2$. Then

$$
\sum p_2 = 0, \qquad \sum x\, p_2(x) = (-2)(2) + (-1)(-1) + (1)(-1) + (2)(2) = 0,
$$

and

$$
\sum p_2(x)^2 = 4 + 1 + 4 + 1 + 4 = 14.
$$

The three columns $1$, $x$, and $p_2$ are pairwise orthogonal. For $y = 10, 0, 1, 8, 16$,

$$
\begin{aligned}
a_0 &= \dfrac{\sum y}{5} = 7, \\
a_1 &= \dfrac{\sum xy}{\sum x^2} = \dfrac{20}{10} = 2, \\
a_2 &= \dfrac{\sum p_2(x)\, y}{14}
= \dfrac{2\cdot 10 + (-2)\cdot 1 + (-1)\cdot 8 + 2\cdot 16}{14}
= \dfrac{42}{14} = 3.
\end{aligned}
$$

Translate back to monomial coefficients. $a_0 + a_1 x + a_2(x^2 - 2) = (a_0 - 2a_2) + a_1 x + a_2 x^2$, so

$$
b_0 = 7 - 6 = 1, \qquad b_1 = 2, \qquad b_2 = 3,
$$

the quadratic already fitted. Orthogonalizing the columns did not change the fitted polynomial. It made each coefficient independent of the others: adding $p_2$ does not revise $a_0$ or $a_1$.


In [1]:
# Three orthogonal columns, three separate divisions, then the monomial translation.
from fractions import Fraction
x = [-2, -1, 0, 1, 2]
y = [10, 0, 1, 8, 16]
p2 = [xi ** 2 - 2 for xi in x]
a0 = Fraction(sum(y), 5)
a1 = Fraction(sum(xi * yi for xi, yi in zip(x, y)), sum(xi ** 2 for xi in x))
a2 = Fraction(sum(p * yi for p, yi in zip(p2, y)), sum(p * p for p in p2))
print("orthogonal coefficients", a0, a1, a2)
print("p2 column", p2)
assert p2 == [2, -1, -2, -1, 2]
assert sum(p2) == 0
assert sum(xi * p for xi, p in zip(x, p2)) == 0
assert (a0, a1, a2) == (7, 2, 3)
assert (a0 - 2 * a2, a1, a2) == (1, 2, 3)


orthogonal coefficients 7 2 3
p2 column [2, -1, -2, -1, 2]


## Ridge on the orthogonal coefficients

Penalize the slope and the curvature, and leave the intercept alone. With orthogonal columns the penalized normal equations stay diagonal. For a penalty $\lambda$,

$$
a_1(\lambda) = \dfrac{20}{10 + \lambda}, \qquad
a_2(\lambda) = \dfrac{42}{14 + \lambda}, \qquad
a_0(\lambda) = 7.
$$

At $\lambda = 0$ this is ordinary least squares. At $\lambda = 14$,

$$
a_2 = \dfrac{42}{28} = \dfrac{3}{2}, \qquad a_1 = \dfrac{20}{24} = \dfrac{5}{6}.
$$

At $\lambda = 7$, $a_2 = 42/21 = 2$. Shrinkage moves each orthogonal coefficient toward zero by the factor $s_j/(s_j + \lambda)$, where $s_j$ is that column's sum of squares. The intercept remains $\bar y$ because its column was not penalized and is orthogonal to the others.

The prediction at $x = 0$ is not $\bar y$. There $p_2(0) = -2$, so

$$
\widehat y(0) = 7 + a_2(-2).
$$

Ordinary least squares gives $7 - 6 = 1$. Ridge with $a_2 = 3/2$ gives $7 - 3 = 4$. The average of the five fitted values remains $7$, because the average of $p_2$ is zero. Shrinkage preserves that average and still moves the center point.


In [2]:
# Ridge divides each orthogonal sum of squares by itself plus lambda.
from fractions import Fraction
a0 = 7
a1 = Fraction(20, 10 + 14)
a2 = Fraction(42, 14 + 14)
center = a0 + a2 * (-2)
fitted = [a0 + a1 * xi + a2 * (xi ** 2 - 2) for xi in (-2, -1, 0, 1, 2)]
print("lambda 14", a0, a1, a2)
print("prediction at 0", center)
print("average fitted", Fraction(sum(fitted), 5))
assert a1 == Fraction(5, 6) and a2 == Fraction(3, 2)
assert center == 4
assert Fraction(sum(fitted), 5) == 7
assert Fraction(42, 14 + 0) == 3


lambda 14 7 5/6 3/2
prediction at 0 4
average fitted 7


## Pitfall

Penalizing the raw monomial coefficients $b_0, b_1, b_2$ is not the same calculation. Those columns are not orthogonal, and $b_0$ is the prediction at $x = 0$ rather than the average response. A penalty on $b_0$ pulls the prediction at the origin toward zero. The orthogonal version above refuses that pull: $a_0$ stays at $7$ for every $\lambda$. Which coefficients are penalized is part of the model, just as the choice of origin was.


In [3]:
# The unpenalized orthogonal intercept does not depend on lambda.
intercepts = [7 for _ in (0, 7, 14, 100)]
print("intercept across lambda", intercepts)
assert len(set(intercepts)) == 1


intercept across lambda [7, 7, 7, 7]


## Summary

- $p_2(x) = x^2 - 2$ is orthogonal to $1$ and to $x$. The coefficients are $7$, $2$, and $3$, and they translate back to $1 + 2x + 3x^2$.
- Ridge gives $a_1 = 20/(10+\lambda)$ and $a_2 = 42/(14+\lambda)$. At $\lambda = 14$ the curvature coefficient is $3/2$.
- The average fitted value stays at $\bar y$. The fitted value at $x = 0$ moves, because $p_2(0) = -2$.
